# Časť 2, Téma 2, Lab A: Analýza spotrby na získanie hesla (MAIN)

---
POZNÁMKA: Toto cvičenie odkazuje na niektoré (komerčné) školiace materiály na stránke ChipWhisperer.io. Cvičenie môžete voľne spúšťať a používať v súlade s licenciou open source (vrátane použitia vo vašich vlastných kurzoch, ak ich distribuujete podobným spôsobom), musíte však zachovať odkaz na tento zdroj. Zvážte účasť na našom školiacom kurze, aby ste si mohli vychutnať plnohodnotný zážitok.

---

**Zhrnutie:** *Tento návod vás zoznámi s technikou „breaking devices“ prostredníctvom určovania, kedy zariadenie vykonáva určité operácie. Naše cieľové zariadenie bude vykonávať jednoduchú kontrolu hesla a my vám ukážeme, ako vykonať základnú analýzu spotreby energie.*

**Čo sa naučíme:**

* Ako možno využiť spotrebu energie na určenie časových informácií.
* Vytváranie grafov viacerých iterácií pri menení vstupných údajov s cieľom nájsť zaujímavé miesta.
* Využitie rozdielov v priebehoch signálov na nájdenie zaujímavých miest.
* Zaznamenávanie spotreby energie pomocou hardvéru ChipWhisperer (len hardvér)


## Predpoklady

Počkajte! Než budete pokračovať, skontrolujte, či ste absolvovali nasledujúce tutoriály:

* ☑ Úvod do Jupyter Notebooku (mali by ste vedieť kresliť grafy a spúšťať bloky).
* ☑ Úvod do SCA101 (mali by ste mať predstavu o tom, ako spustiť verzie špecifické pre konkrétny hardvér).

Najskor skompilujeme a nahráme firmvér na cieľ

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
SS_VER = 'SS_VER_2_1'



Tento kód pripojí osciloskop a vykoná základné nastavenia. Teraz na to použijeme špeciálny nastavovací skript. Tento skript obsahuje príkazy, ktoré sme predtým spúšťali samostatne.

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

Nasledujúci kód skompiluje firmvér pre cieľové zariadenie.

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../firmware/mcu/basic-passwdcheck
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j


Nakoniec už zostáva len naprogramovať zariadenie, čo sa dá urobiť pomocou nasledujúceho riadku:

In [ ]:
cw.program_target(scope, prog, "../../firmware/mcu/basic-passwdcheck/basic-passwdcheck-{}.hex".format(PLATFORM))

Aby sme uľahčili komunikáciu s hardvérom, definujme funkciu, ktorá vyskúša heslo a vráti záznam o spotrebe energie:

In [ ]:
def cap_pass_trace(pass_guess):
    reset_target(scope)
    num_char = target.in_waiting()
    while num_char > 0:
        target.read(num_char, 10)
        time.sleep(0.01)
        num_char = target.in_waiting()

    scope.arm()
    target.write(pass_guess)
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')

    trace = scope.get_last_trace()
    return trace

V zázname tiež nepotrebujeme všetkých 5000 vzoriek, ktoré sú nastavené ako predvolené. Pre väčšinu cieľov je 3000 dobrým východiskovým bodom:

In [ ]:
scope.adc.samples = 3000

## Úvod

Čo s tým teda môžeme urobiť? Najprv vám trochu pomôžem a prezradím, že máme prednastavené heslo, ktoré začína na `h` a má 5 znakov. To je však zatiaľ jediná nápoveda – čo s tým urobíte? Najprv skúsme nakresliť graf, v ktorom porovnáme `h` s niečím iným.

Ak potrebujete pripomenúť, ako vytvoriť graf, pozrite si časť o matplotlib v zošite **Úvod do Jupyteru**.

Nasledujúca bunka vám ukáže, ako zachytiť jednu krivku výkonu s `h` odoslaným ako heslo. Odtiaľ:

1. Skúste pridať kód na vytvorenie grafu a pozrite sa, ako to vyzerá.
2. Pošlite do zariadenia rôzne heslá. Momentálne sa budeme zaoberať len rozdielom medzi heslom začínajúcim na `h` a niečím iným.
3. Vytvorte grafy rôznych priebehov.


In [ ]:
#Príklad - capture 'h' - ukonči to s '\n' pretože to očakáva seriový protokol
trace_h = cap_pass_trace("h\n")

print(trace_h)

# ###################
# Sem pridaj svoj kód
# ###################
raise NotImplementedError("Sem pridaj svoj kód, a vymaž tento riadok.")

V prípade problémov so grafovými knižnicami odporúčam skúsiť:
``` py
%matplotlib inline
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4)) //velkosť grafu
plt.plot(trace_h, label="h", alpha=0.7) //prvy argument su data ktore chceme mat v grafe, druhy je oznacenie  a treti ako to bude mat priehladnost
plt.plot(other_trace, label="x", alpha=0.7) //sem vlož to s čím porovnávaš
plt.legend()
plt.show()
```


Ak si všetko správne tak si môžeš všimnúť ze pri jednom zázname existuje peak ktorý sa dosahuje v iný moment, takže vieme že pri správnom prvom znaku sa kód správa odlišne

OK, zaujímavé – čo ďalej? Vykreslime všetky možné znaky hesla, ktoré by sme mohli odoslať.

Naša implementácia hesla rozpoznáva iba znaky zo zoznamu `abcdefghijklmnopqrstuvwxyz0123456789`, takže sa zatiaľ obmedzíme na tieto platné znaky.

Napíšte do nasledujúceho bloku kód, ktorý implementuje nasledujúci algoritmus:

for CHARACTER in LIST_OF_VALID_CHARACTERS:
    trace = cap_pass_trace(CHARACTER + „\n“)
    plot(trace)
        
Vyššie uvedený kód nie je úplne platný – upravte ho tak, aby fungoval! Možno si tiež všimnete, že krivky sú príliš dlhé – možno budete chcieť vytvoriť užší graf, ktorý zobrazuje len prvých, povedzme, 500 vzoriek krivky výkonu.

---
📝**Poznámka k vykresľovaniu**

Ak na vykresľovanie používate `matplotlib`, možno budete musieť na začiatku pridať `plt.figure()`, aby ste vytvorili nový obrázok. Inak sa môže stať, že sa váš graf ocitne na obrázku vyššie! Ak grafy nevidíte, niekedy je potrebné na konci pridať `plt.show()`.

---

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Sem pridaj svoj kód, a vymaž tento riadok.")

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    %matplotlib widget
    import matplotlib.pyplot as plt
    import mplcursors

    plt.figure(figsize=(10,4))
    lines = []
    LIST_OF_VALID_CHARACTERS = "abcdefghijklmnopqrstuvwxyz0123456789"
    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace(CHARACTER + "\n")
        line, = plt.plot(trace)
        line.set_label(CHARACTER)
        lines.append(line)

    # Zapnutie labelov pri hover nad grafom
    cursor = mplcursors.cursor(lines, hover=True)
    cursor.connect("add", lambda sel: sel.annotation.set_text(sel.artist.get_label()))

    plt.show()
</code></pre>
</details>

Konečným výsledkom by malo byť, že ak obraz zväčšíte, uvidíte miesto, kde jedna „odchýlka“ nesleduje dráhu všetkých ostatných kriviek. To je skvelá správa, pretože to znamená, že z analýzy výkonu sa o systéme niečo dozvieme.

<img src="img/spa_password_list_char1.png" alt="SPA of Power Analysis against all inputs" width="450"/>

Pomocou vašej slučky môžete tiež skúsiť upraviť analýzu tak, aby zachytila správny „prvý“ znak a potom každý druhý nesprávny znak. Vidíte rozdiel, ktorý by ste mohli zistiť?

Pseudokód by vyzeral približne takto:
```py
    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace(„h“ + CHARACTER + „\n“)
        plot(trace)
```
Vyskúšajte to vo vašom predchádzajúcom bloku kódu a potom sa pokúsme tento útok automatizovať, aby sme lepšie pochopili údaje.

## Automatizovanie útokov

Na úvod – automatizujeme útok zameraný na **jeden** znak hesla. Keďže heslo nepoznáme (predpokladajme to), použijeme stratégiu, pri ktorej porovnáme všetky možné vstupy navzájom.

Jednoduchým spôsobom, ako to urobiť, je použiť niečo, o čom vieme, že nemôže byť súčasťou platného hesla. Pokiaľ sa to spracováva rovnakým spôsobom, bude to fungovať bez problémov. Preto zatiaľ použijeme heslo `0x00` (t. j. nulový bajt). Môžeme porovnať nulový bajt so spracovaním niečoho iného:

In [ ]:
ref_trace = cap_pass_trace("\x00\n")[0:1000]
fig = cw.plot(ref_trace)
other_trace = cap_pass_trace("c\n")[0:1000]
fig *= cw.plot(other_trace)
fig

Týmto sa vykreslí krivka so vstupom „\x00“ – nulovým heslom! Ide o neplatný znak, ktorý sa zrejme spracováva rovnako ako akékoľvek iné neplatné heslo.

Urobme to trochu názornšie a vykreslime rozdiel medzi známou referenčnou hodnotou a všetkými ostatnými zachytenými hodnotami. Musíte napísať kód, ktorý bude fungovať približne takto:
``` py
ref_trace = cap_pass_trace( "\x00\n")

for CHARACTER in LIST_OF_VALID_CHARACTERS:
    trace = cap_pass_trace(CHARACTER + "\n")
    plot(trace - ref_trace)
```
Možno budete musieť tento kód opäť trochu upraviť, napríklad pridať kód na vytvorenie nového grafu (figure()). Všímajte si tiež, ako som v uvedenom príklade znížil počet vzoriek.

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Sem pridaj svoj kód, a vymaž tento riadok.")

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    %matplotlib widget
    import matplotlib.pyplot as plt
    import mplcursors

    plt.figure(figsize=(10,4))
    lines = []
    LIST_OF_VALID_CHARACTERS = "abcdefghijklmnopqrstuvwxyz0123456789"
    ref_trace = cap_pass_trace( "\x00\n")
    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace(CHARACTER + "\n")
        line, = plt.plot(trace-ref_trace)
        line.set_label(CHARACTER)
        lines.append(line)

    # Zapnutie labelov pri hover nad grafom
    cursor = mplcursors.cursor(lines, hover=True)
    cursor.connect("add", lambda sel: sel.annotation.set_text(sel.artist.get_label()))

    plt.show()
</code></pre>
</details>

Super – dúfam, že teraz vidíte jeden zásadný „rozdiel“. Malo by to vyzerať približne takto:
    
<img src="img/spa_password_diffexample.png" alt="SPA s rozdielom" width="450"/>

Čo teraz? Nastavme to tak, aby systém automaticky rozpoznal takýto veľký rozdiel. Skúste využiť funkcie np.abs() a np.sum().

Prvá z nich vypočíta absolútne hodnoty:
```py
import numpy as np
np.abs([-1, -3, 1, -5, 6])

    Out[]: array([1, 3, 1, 5, 6])
```

Druhá spočíta všetky čísla.

```py
import numpy as np    
np.sum([-1, -3, 1, -5, 6])

    Out[]: -2
```
Ak použijete len `np.sum()`, kladné a záporné rozdiely sa navzájom zrušia – preto je lepšie použiť napríklad `np.sum(np.abs(DIFF))`, aby ste získali relevantnú hodnotu, ktorá vyjadruje, ako „tesný“ bol výsledok.

In [ ]:
import numpy as np
np.abs([-1, -3, 1, -5, 6])

In [ ]:
import numpy as np
np.sum([-1, -3, 1, -5, 6])

In [ ]:
np.sum(np.abs(<COPY DATA HERE>))

Vezmite uvedenú slučku a upravte ju tak, aby vypísala indikátor toho, do akej miery sa zhoduje s vašou krivkou. Niečo podobné ako nasledujúci kód by malo fungovať:

```py
ref_trace = cap_pass_trace( "\x00\n")

for CHARACTER in LIST_OF_VALID_CHARACTERS:
    trace = cap_pass_trace(CHARACTER + "\n")
    diff = SUM(ABS(trace - ref_trace))

    print("{:1} diff = {:2}".format(CHARACTER, diff))
```


In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Sem pridaj svoj kód, a vymaž tento riadok.")

Teraz tá ľahšia časť – upravte svoj vyššie uvedený kód tak, aby automaticky vytlačil správny znak hesla. To by sa malo dosiahnuť porovnaním premennej `diff` – na základe vytlačených znakov by ste mali nájsť jeden, ktorý je „vyššie“ ako ostatné. Nastavte prah na rozumnú hodnotu (napríklad na základe jedného behu by som mohol použiť hodnotu 25,0).

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    %matplotlib widget
    import matplotlib.pyplot as plt
    import mplcursors
    import numpy as np

    plt.figure(figsize=(10,4))
    lines = []
    LIST_OF_VALID_CHARACTERS = "abcdefghijklmnopqrstuvwxyz0123456789"
    ref_trace = cap_pass_trace( "\x00\n")
    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace(CHARACTER + "\n")
        diff = np.sum(np.abs(trace-ref_trace))
        line, = plt.plot(trace-ref_trace)
        line.set_label(CHARACTER)
        lines.append(line)
        print("{:1} diff = {:2}".format(CHARACTER, diff))

    # Zapnutie labelov pri hover nad grafom
    cursor = mplcursors.cursor(lines, hover=True)
    cursor.connect("add", lambda sel: sel.annotation.set_text(sel.artist.get_label()))

    plt.show()
</code></pre>
</details>

Na záver – dokončime to. Namiesto toho, aby sme sa zamerali na jeden znak, musíme postupne vyskúšať každý znak.

Ak sa vrátite k zaznamenávaniu rozdielov, môžete vyskúšať kombináciu správneho prvého znaku a nesprávneho druhého znaku. Základná myšlienka je presne rovnaká ako predtým, ale teraz prechádzame cyklus päťkrát a heslo zostavujeme tak, že každý znak vyskúšame metódou hrubou silou.

Pozrite sa na nasledujúci základný pseudokód:
```py
guessed_pw = "" #sem ulozime uz zatial uhadnute heslo

do a loop 5 times (max password size):
    
    ref_trace = capture power trace(guessed_pw + "\x00\n")
    
    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = capture power trace (guessed_pw + CHARACTER + newline)
        diff = SUM(ABS(trace - ref_trace))
        
        if diff > THRESHOLD:
            
            guessed_pwd += c
            print(guessed_pw)
            
            break
```

<details>
<summary>Nápoveda</summary>
Nezabudni vzdy vytvorit novu referenčnu vzorku po kazdom uhadnutom znaku
</details>

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    %matplotlib widget
    import matplotlib.pyplot as plt
    import mplcursors
    import numpy as np

    plt.figure(figsize=(10,4))
    lines = []
    LIST_OF_VALID_CHARACTERS = "abcdefghijklmnopqrstuvwxyz0123456789"
    password = ""


    for _ in range(6):
        ref_trace = cap_pass_trace( password+"\x00\n")
        for CHARACTER in LIST_OF_VALID_CHARACTERS:
            trace = cap_pass_trace(password+CHARACTER + "\n")
            diff = np.sum(np.abs(trace-ref_trace))
            line, = plt.plot(trace-ref_trace)
            line.set_label(CHARACTER)
            lines.append(line)
            #print("{:1} diff = {:2}".format(CHARACTER, diff))
            if(diff>100):
                password += CHARACTER
                print(password)
                break
                

    # Zapnutie labelov pri hover nad grafom
    cursor = mplcursors.cursor(lines, hover=True)
    cursor.connect("add", lambda sel: sel.annotation.set_text(sel.artist.get_label()))

    plt.show()
</code></pre>
</details>

Ak si to správne naprogramoval mal by si mať takýto výstup:

    h
    h0
    h0p
    h0px
    h0px3

Ak áno gratulujem, zvládol si to


In [ ]:
scope.dis()
target.dis()

---
<small>UPOZORNENIE: Tento materiál je chránený autorskými právami (C) NewAE Technology Inc., 2015–2020. ChipWhisperer je ochranná známka spoločnosti NewAE Technology Inc., na ktorú si táto spoločnosť uplatňuje práva vo všetkých jurisdikciách a ktorá je registrovaná minimálne v Spojených štátoch amerických, Európskej únii a Čínskej ľudovej republike.

Návody odvodené z našej open-source práce musia byť uverejnené pod príslušnou open-source licenciou a upozornenie na zdroj musí byť *jasne zobrazené*. Iba pôvodní držitelia autorských práv môžu udeľovať licencie alebo povoľovať ďalšiu distribúciu – hoci spoločnosť NewAE Technology Inc. vlastní autorské práva k mnohým návodom, repozitár github obsahuje príspevky komunity, ktoré nemôže licencovať za špeciálnych podmienok a **musia** byť udržiavané ako open-source verzia. V prípade potreby prosím kontaktujte NewAE Technology Inc ohľadom špeciálnych povolení (ak je to možné)

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>